In [82]:
import numpy as np
from safetensors.numpy import load_file
from transformers import AutoTokenizer

In [83]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


In [84]:
class GELU:
    def __call__(self, x):
        return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

    def __str__(self):
        return ""

In [85]:
class Linear:
    def __init__(self, weights, bias) -> None:
        self.weights = weights
        self.bias = bias
    
    def __call__(self, x):
        return x @ self.weights + self.bias

    def __str__(self):
        return ""


In [86]:
class LayerNorm:
  def __init__(self, weight, bias, eps=1e-5):
    self.weight = weight
    self.bias = bias
    self.eps = eps

  def __call__(self, x):
      mean = np.mean(x, axis=-1, keepdims=True)
      var = np.var(x, axis=-1, keepdims=True)
      
      x_norm = (x - mean) / np.sqrt(var + self.eps)

      return self.weight * x_norm + self.bias


In [87]:
# transformer.h.0.attn.c_attn.bias
# transformer.h.0.attn.c_attn.weight
# transformer.h.0.attn.c_proj.bias
# transformer.h.0.attn.c_proj.weight

class Attention:
    def __init__(self, attn_weights, attn_bias, proj_weights, proj_bias) -> None:
        self.attn_weights = attn_weights
        self.attn_bias = attn_bias
        self.proj_weights = proj_weights
        self.proj_bias = proj_bias

    def __call__(self, x):
        qkv = x @ self.attn_weights + self.attn_bias

        q, k, v = np.split(qkv, 3, axis=-1)

        scores = q @ k.T
        scores = scores / np.sqrt(k.shape[-1])
        scores = softmax(scores)

        out = scores @ v

        out = out @ self.proj_weights + self.proj_bias

        return out


    def __str__(self):
        return ""


In [88]:
class MultiHeadAttention:
    def __init__(self, attn_weights, attn_bias, proj_weights, proj_bias, num_heads=8) -> None:
        self.attn_weights = attn_weights
        self.attn_bias = attn_bias
        self.proj_weights = proj_weights
        self.proj_bias = proj_bias
        self.num_heads = num_heads

    def __call__(self, x):
        seq_len = x.shape[0]
        hidden_dim = x.shape[1]  # Should be 512
        head_dim = hidden_dim // self.num_heads  # 512 // 8 = 64
        
        # 1. Project input to Q, K, V
        qkv = x @ self.attn_weights + self.attn_bias  # (seq_len, 512*3)
        q, k, v = np.split(qkv, 3, axis=-1)  # Each: (seq_len, 512)
        
        # 2. Split into multiple heads
        # Reshape from (seq_len, 512) to (seq_len, num_heads, head_dim)
        q = q.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        k = k.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        v = v.reshape(seq_len, self.num_heads, head_dim)  # (seq_len, 8, 64)
        
        # 3. Transpose to (num_heads, seq_len, head_dim) for batch processing
        q = q.transpose(1, 0, 2)  # (8, seq_len, 64)
        k = k.transpose(1, 0, 2)  # (8, seq_len, 64)
        v = v.transpose(1, 0, 2)  # (8, seq_len, 64)
        
        # 4. Compute attention scores for all heads
        scores = q @ k.transpose(0, 2, 1)  # (8, seq_len, seq_len)
        scores = scores / np.sqrt(head_dim)  # Scale by sqrt(64)
        
        # 5. Apply causal mask (prevent attending to future tokens)
        causal_mask = np.tril(np.ones((seq_len, seq_len)))  # Lower triangular matrix
        scores = np.where(causal_mask == 0, -1e10, scores)  # Mask future positions
        
        # 6. Apply softmax to get attention weights
        attn_weights = softmax(scores, axis=-1)  # (8, seq_len, seq_len)
        
        # 7. Apply attention weights to values
        output = attn_weights @ v  # (8, seq_len, 64)
        
        # 8. Concatenate heads back together
        output = output.transpose(1, 0, 2)  # (seq_len, 8, 64)
        output = output.reshape(seq_len, hidden_dim)  # (seq_len, 512)
        
        # 9. Final projection
        output = output @ self.proj_weights + self.proj_bias
        
        return output

    def __str__(self):
        return ""

In [89]:
class Transformer:
    def __init__(self, model) -> None:
        self.model = model
        self.attention_blocks = []
        self.ffn_blocks = []

        for i in range(4):
            # Attention block (LayerNorm + Attention)
            ln_1_weight = self.model[f"transformer.h.{i}.ln_1.weight"]
            ln_1_bias = self.model[f"transformer.h.{i}.ln_1.bias"]
            attn_c_attn_weight = self.model[f"transformer.h.{i}.attn.c_attn.weight"]
            attn_c_attn_bias = self.model[f"transformer.h.{i}.attn.c_attn.bias"]
            attn_c_proj_weight = self.model[f"transformer.h.{i}.attn.c_proj.weight"]
            attn_c_proj_bias = self.model[f"transformer.h.{i}.attn.c_proj.bias"]
            
            self.attention_blocks.append({
                'ln': LayerNorm(ln_1_weight, ln_1_bias),
                'attn': MultiHeadAttention(attn_c_attn_weight, attn_c_attn_bias, attn_c_proj_weight, attn_c_proj_bias)
            })

            # FFN block (LayerNorm + Linear + GELU + Linear)
            ln_2_weight = self.model[f"transformer.h.{i}.ln_2.weight"]
            ln_2_bias = self.model[f"transformer.h.{i}.ln_2.bias"]
            ffn_1_weight = self.model[f"transformer.h.{i}.mlp.c_fc.weight"]
            ffn_1_bias = self.model[f"transformer.h.{i}.mlp.c_fc.bias"]
            ffn_2_weight = self.model[f"transformer.h.{i}.mlp.c_proj.weight"]
            ffn_2_bias = self.model[f"transformer.h.{i}.mlp.c_proj.bias"]
            
            self.ffn_blocks.append({
                'ln': LayerNorm(ln_2_weight, ln_2_bias),
                'linear1': Linear(ffn_1_weight, ffn_1_bias),
                'gelu': GELU(),
                'linear2': Linear(ffn_2_weight, ffn_2_bias)
            })

    def __call__(self, x):
        for attn_block, ffn_block in zip(self.attention_blocks, self.ffn_blocks):
            # Attention with residual
            attn_out = attn_block['attn'](attn_block['ln'](x))
            x += attn_out  # ← Residual connection
            
            # FFN with residual
            ffn_out = ffn_block['linear2'](ffn_block['gelu'](ffn_block['linear1'](ffn_block['ln'](x))))
            x += ffn_out  # ← Residual connection

        # final layer norm
        x = LayerNorm(self.model["transformer.ln_f.weight"], self.model["transformer.ln_f.bias"])(x)

        logits = x @ self.model["transformer.wte.weight"].T
        token_ids = np.argmax(logits, axis=-1)

        return int(token_ids[-1])

In [90]:
tokenizer = AutoTokenizer.from_pretrained("erwanf/gpt2-mini", cache_dir="./data")
model_weights = load_file("./data/model.safetensors")

model = Transformer(model_weights)
print(model_weights.keys())

def generate(input_text: str):
    input_tokens = tokenizer.encode(input_text)
    token_embeddings = model_weights["transformer.wte.weight"][input_tokens]
    pos_embeddings = model_weights["transformer.wpe.weight"][:len(input_tokens)]
    embeddings = token_embeddings + pos_embeddings
    next_token = model(embeddings)
    return tokenizer.decode(next_token)

print(generate("OpenAI is a company that makes AI models"))

dict_keys(['transformer.h.0.attn.c_attn.bias', 'transformer.h.0.attn.c_attn.weight', 'transformer.h.0.attn.c_proj.bias', 'transformer.h.0.attn.c_proj.weight', 'transformer.h.0.ln_1.bias', 'transformer.h.0.ln_1.weight', 'transformer.h.0.ln_2.bias', 'transformer.h.0.ln_2.weight', 'transformer.h.0.mlp.c_fc.bias', 'transformer.h.0.mlp.c_fc.weight', 'transformer.h.0.mlp.c_proj.bias', 'transformer.h.0.mlp.c_proj.weight', 'transformer.h.1.attn.c_attn.bias', 'transformer.h.1.attn.c_attn.weight', 'transformer.h.1.attn.c_proj.bias', 'transformer.h.1.attn.c_proj.weight', 'transformer.h.1.ln_1.bias', 'transformer.h.1.ln_1.weight', 'transformer.h.1.ln_2.bias', 'transformer.h.1.ln_2.weight', 'transformer.h.1.mlp.c_fc.bias', 'transformer.h.1.mlp.c_fc.weight', 'transformer.h.1.mlp.c_proj.bias', 'transformer.h.1.mlp.c_proj.weight', 'transformer.h.2.attn.c_attn.bias', 'transformer.h.2.attn.c_attn.weight', 'transformer.h.2.attn.c_proj.bias', 'transformer.h.2.attn.c_proj.weight', 'transformer.h.2.ln_1.bia

In [94]:
text = "User: Hello, how are you? Assistant:"

for i in range(20):
    text += generate(text)
    print(text)

User: Hello, how are you? Assistant: I
User: Hello, how are you? Assistant: I'm
User: Hello, how are you? Assistant: I'm a
User: Hello, how are you? Assistant: I'm a big
User: Hello, how are you? Assistant: I'm a big fan
User: Hello, how are you? Assistant: I'm a big fan of
User: Hello, how are you? Assistant: I'm a big fan of the
User: Hello, how are you? Assistant: I'm a big fan of the game
User: Hello, how are you? Assistant: I'm a big fan of the game,
User: Hello, how are you? Assistant: I'm a big fan of the game, and
User: Hello, how are you? Assistant: I'm a big fan of the game, and I
User: Hello, how are you? Assistant: I'm a big fan of the game, and I'm
User: Hello, how are you? Assistant: I'm a big fan of the game, and I'm a
User: Hello, how are you? Assistant: I'm a big fan of the game, and I'm a big
User: Hello, how are you? Assistant: I'm a big fan of the game, and I'm a big fan
User: Hello, how are you? Assistant: I'm a big fan of the game, and I'm a big fan of
User: Hello